In [2]:
%nvidia-smi

UsageError: Line magic function `%nvidia-smi` not found.


In [ ]:
%ls .config

active_config
config_sentinel
configurations/
default_configs.db
gce
hidden_gcloud_config_universe_descriptor_data_cache_configs.db
logs/


### Imports & Utils

In [13]:
%pip install rasterio
import rasterio
import subprocess
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from torch.cuda.amp import autocast, GradScaler
import time
import gc
import numpy as np
import sys
from collections.abc import Callable, Collection, Iterable, Mapping, Sequence
from types import TracebackType
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from torchvision.io import read_image
import torchvision.transforms as T
import torch.nn.functional as F
from torch.utils.data import DataLoader
from rasterio.errors import NotGeoreferencedWarning
import json
import warnings
import os
import copy
from dotenv import load_dotenv
load_dotenv()
def wandb_login():
    wandb_key = os.getenv("WANDB_API_KEY")
    subprocess.run(["wandb", "login", wandb_key])

wandb_login()

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
# new
class LCDataset(torch.utils.data.Dataset):
    def __init__(self, img_paths, label_paths, pair_transform=None, transform=None, target_transform=None):
        self.img_paths = img_paths
        self.label_paths = label_paths
        self.pair_transform_attr = pair_transform
        self.transform = transform
        self.target_transform = target_transform

    def pair_transform(self, image, mask):
        both_images = torch.concat(tensors=(image, mask), dim=0) #(Ci + Cm, W, H)
        both_images = self.pair_transform_attr(both_images)
        image = both_images[:3, :, :] # (Ci, W, H)
        mask = both_images[3:, :, :] # (Cm, W, H)
        return image, mask

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label_path = self.label_paths[idx]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", NotGeoreferencedWarning)
            image = rasterio.open(img_path).read()
            image = torch.tensor(image)
            label = rasterio.open(label_path).read()
            label = torch.tensor(label)
        if self.pair_transform_attr:
            image, label = self.pair_transform(image, label)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)

        # Normalize image to 0 - 1
        image = (image - image.min()) / (image.max() - image.min()) # normalize
        return image, label

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

def split_into_patches(img, label, patch_size):
    """
    Given an image-segmentation label pair, split the image and label into patches according to patch_size.
    """
    # assuming img and label are of shape (N, C, W, H)
    N, C, W, H = img.shape
    img_patches = img.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size) # non-overlapping (N, C, num_patches_w, num_patches_h, W, H)
    N, C, num_patches_w, num_patches_h, W, H = img_patches.shape
    img_patches = img_patches.permute(0, 2, 3, 1, 4, 5).contiguous()
    img_patches = img_patches.contiguous().view(N*num_patches_w*num_patches_h, C, patch_size, patch_size) # idk (N, C, W, H)
    label_patches = label.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size) # non-overlapping
    label_patches = label_patches.permute(0, 2, 3, 1, 4, 5).contiguous()
    label_patches = label_patches.contiguous().view(N*num_patches_w*num_patches_h, C, patch_size, patch_size) # idk
    return img_patches, label_patches

def convert_to_label(label_rgb, color_map, num_classes=6): # slow
    # assuming label is of shape (N, 3, W, H)
    # convert into class labels -> (N, C, W, H)
    N, C, W, H = label_rgb.shape
    label_rgb = label_rgb.permute(0, 2, 3, 1) # (N, W, H, 3)
    label_tensor = torch.zeros((N, W, H), dtype=torch.long)
    for color, class_idx in color_map.items():
        mask = (label_rgb == torch.tensor(color)).all(dim=-1)
        label_tensor[mask] = class_idx
    one_hot = torch.nn.functional.one_hot(label_tensor, num_classes=num_classes)  # Shape: (N, W, H, num_classes)
    one_hot = one_hot.permute(0, 3, 1, 2).float()  # Shape: (N, num_classes, W, H)
    return one_hot

def label_to_img(label, color_map):
    # assuming label is in one-hot-encoded form and in cpu
    N, C, W, H = label.shape # (N, C, W, H)
    class_indices = torch.argmax(label, dim=1) # (N, W, H)
    rgb_img = torch.zeros((N, 3, W, H), dtype=torch.uint8)
    for color, class_idx in color_map.items():
        mask = class_indices == class_idx # true if the class indices tensor matches the current class_idx
        mask = mask.unsqueeze(1)  # Shape: (N, 1, W, H) for broadcasting

        # Convert color tuple to a tensor and reshape for broadcasting
        color_tensor = torch.tensor(color, dtype=torch.uint8).view(1, 3, 1, 1)

        # Apply color where mask is True
        rgb_img = torch.where(mask, color_tensor, rgb_img)
    return rgb_img


def pad_to_6144(img):
    # img -> tensor of shape (N, C, W, H)
    N, C, W, H = img.shape
    pad_w = 6144 - W
    pad_h = 6144 - H

    padding = (0, pad_w, 0, pad_h)
    img_padded = F.pad(img, padding, mode='reflect')
    return img_padded

def crop_to_6000(img):
    return img[:, :, :6000, :6000]

def inference(model, img, label, loss_fn, patch_size, patch_stride, optimizer=None, training=False, invert_loss=False, device="cpu", verbose=False,
              optimize_memory=False, scaler=None):
    '''
    Performs sliding window inference over a batch. Set training to False for validation and testing. Make sure optimizer.zero_grad() is called before this function if training=True.
    Returns the inference loss, image, prediction, and label.
    Moves image and label to device.
    '''
    batch_loss = 0.0
    num_inferences = 0

    img = pad_to_6144(img)
    label = pad_to_6144(label)
    N, C, W, H = label.shape
    pred_full = torch.zeros((N, C, W, H), dtype=torch.float32)
    count_map = torch.zeros((N, C, W, H), dtype=torch.float32)

    for w in range(0, img.shape[2]-patch_size+1, patch_stride):
        for h in range(0, img.shape[3]-patch_size+1, patch_stride):
            img_patch = img[:, :, w:w+patch_size, h:h+patch_size].detach().clone()
            label_patch = label[:, :, w:w+patch_size, h:h+patch_size].detach().clone()
            img_patch, label_patch = img_patch.to(device), label_patch.to(device)
            pred_patch = None

            if not training:
                pred_patch = model(img_patch)
                loss = 1 - loss_fn(pred_patch, label_patch) if invert_loss else loss_fn(pred_patch, label_patch)
            else:
                if optimize_memory and scaler is not None: # training using automatic mixed precision (AMP) with autocast():
                    with torch.amp.autocast("cuda"):
                        pred_patch = model(img_patch) # get pred_patch predictions
                        loss = 1 - loss_fn(pred_patch, label_patch) if invert_loss else loss_fn(pred_patch, label_patch) # get loss
                    scaler.scale(loss).backward()
                else: # no AMP
                    pred_patch = model(img_patch) # get pred_patch predictions
                    loss = 1 - loss_fn(pred_patch, label_patch) if invert_loss else loss_fn(pred_patch, label_patch) # get loss
                    loss.backward() # compute gradients
            batch_loss += loss.item()
            num_inferences += 1
            pred_full[:, :, w:w+patch_size, h:h+patch_size] += pred_patch.cpu()
            count_map[:, :, w:w+patch_size, h:h+patch_size] += 1

    pred_full /= count_map
    img = crop_to_6000(img)
    pred_full = crop_to_6000(pred_full)
    label = crop_to_6000(label)
    return (batch_loss/num_inferences), img, pred_full, label


def train_one_epoch(model, data_loader, loss_fn, optimizer, patch_size, color_map=None,
                    invert_loss=False, device="cpu", verbose=False, optimize_memory=False, grad_accum_steps=1, patch_stride=None, num_classes=6):
    '''
    Trains the model. Returns the average epoch loss.
    '''
    model.to(device)
    model.train()
    total_loss = 0.0
    num_inferences = 0
    scaler = torch.amp.GradScaler("cuda") # prevent numerical instability
    optimizer.zero_grad() # zero out gradients before training

    if patch_stride is None: patch_stride = patch_size

    loop = tqdm(data_loader, desc="Training", leave=True)
    for i, (imgs, labels) in enumerate(loop):
        batch_loss = 0.0
        num_inferences = 0
        if color_map is not None:
            labels = convert_to_label(labels, color_map, num_classes)

        batch_loss, _, _, _ = inference(model, imgs, labels, loss_fn=loss_fn, patch_size=patch_size, patch_stride=patch_stride,
                                        optimizer=optimizer,
                                        training=True,
                                        invert_loss=invert_loss, device=device,
                                        verbose=True,
                                        optimize_memory=optimize_memory,
                                        scaler=scaler)

        if (i + 1) % grad_accum_steps == 0 or (i + 1) == len(data_loader):
            if optimize_memory:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()  # Reset gradients after step
            else:
                optimizer.step()
                optimizer.zero_grad()


        total_loss += batch_loss

        running_loss = total_loss/(i+1)
        if verbose: loop.set_postfix(loss=f"{running_loss:.4f}")

    avg_loss = total_loss / len(data_loader)
    return avg_loss


def val_one_epoch(model, data_loader, loss_fn, patch_size=None, color_map=None,
                  invert_loss=False, device="cpu", verbose=False, patch_stride=None, num_classes=6):
    '''
    Disabled training. Returns the average epoch loss.
    '''
    model.to(device)
    model.eval()
    total_loss = 0.0

    if patch_stride is None: patch_stride = patch_size

    with torch.no_grad():
        loop = tqdm(data_loader, desc="Validating", leave=True)
        for i, (imgs, labels) in enumerate(loop):
            if color_map is not None:
                labels = convert_to_label(labels, color_map, num_classes)

            batch_loss, _, _, _ = inference(model, imgs, labels, loss_fn=loss_fn, patch_size=patch_size, patch_stride=patch_stride,
                                        training=False,
                                        invert_loss=invert_loss, device=device,
                                        verbose=True)
            total_loss += batch_loss
            running_loss = total_loss/(i+1)
            if verbose: loop.set_postfix(loss=f"{running_loss:.4f}")

    avg_loss = total_loss / len(data_loader)
    return avg_loss



def test_one_epoch(model, data_loader, loss_fn, patch_size=None, color_map=None,
                  invert_loss=False, device="cpu", verbose=False, patch_stride=None, num_classes=6, return_batch_idx=0):
    '''
    Returns the average epoch loss, images, segmentation labels and predictions. Returned tensors are in device="cpu".
    '''
    model.to(device)
    model.eval()
    batch_imgs = None
    batch_labels = None
    batch_preds = None
    prediction_indices = None
    label_indices = None
    metrics = {"loss" : 0, "score" : 0, "recall" : None, "f1" : None}
    with torch.no_grad():
        loop = tqdm(data_loader, desc="Testing", leave=True)
        for i, (imgs, labels) in enumerate(loop):
            if color_map is not None:
                labels = convert_to_label(labels, color_map, num_classes)

            batch_loss, _, preds, _ = inference(model, imgs, labels, loss_fn=loss_fn, patch_size=patch_size, patch_stride=patch_stride,
                                        training=False,
                                        invert_loss=invert_loss, device=device,
                                        verbose=True)

            metrics["loss"] += batch_loss
            metrics["score"] += 1 - batch_loss

            if prediction_indices is None: prediction_indices = torch.argmax(preds, dim=1) # (N, W, H)
            else: prediction_indices = torch.cat([prediction_indices, torch.argmax(preds, dim=1)], dim=0)

            if label_indices is None: label_indices = torch.argmax(labels, dim=1) # (N, W, H)
            else: label_indices = torch.cat([label_indices, torch.argmax(labels, dim=1)], dim=0)

            running_loss = metrics["loss"]/(i+1)
            if verbose: loop.set_postfix(loss=f"{running_loss:.4f}")

            if i == return_batch_idx:
                batch_imgs = imgs
                batch_labels = labels
                batch_preds = preds

    print("Computing Recall and F1-score...")
    metrics["loss"] /= len(data_loader)
    metrics["score"] /= len(data_loader)
    metrics["recall"] = np.array(recall_score(label_indices.cpu().numpy().flatten(), prediction_indices.cpu().numpy().flatten(), average=None, zero_division=1, labels=[0, 1, 2, 3, 4, 5, 6])).tolist()
    metrics["f1"] = np.array(f1_score(label_indices.cpu().numpy().flatten(), prediction_indices.cpu().numpy().flatten(), average=None, zero_division=1, labels=[0, 1, 2, 3, 4, 5, 6])).tolist()
    batch_imgs, batch_labels, batch_preds = batch_imgs.cpu(), batch_labels.cpu() , batch_preds.cpu()
    try:
        batch_labels = label_to_img(batch_labels, color_map)
        batch_preds = label_to_img(batch_preds, color_map)
    except:
        print("No color map, or label_to_img() failed...")


    return metrics, batch_imgs, batch_labels, batch_preds

def display_images(image_tensor_dicts, image_size=(6, 6)):
    """
    Assume image_tensor_dicts is in the form:
        {
                "title1" : torch.Tensor,
                "title2" : torch.Tensor,
                "title3" : torch.Tensor
        }
    where torch.Tensor is of shape (N, C, W, H).
    Assume N is the same across all tensors.
    """
    title_list = list(image_tensor_dicts.keys())
    tensor_list = list(image_tensor_dicts.values())

    tensors = torch.stack(tensor_list) #(M, N, C, W, H) where M is number of tensors in dict
    col, row = tensors.shape[0], tensors.shape[1]
    for slice_idx in range(row):
        plt.figure(figsize=(image_size[0], image_size[1]*col))
        for tensor_idx in range(col):
            plt.subplot(1, 3, tensor_idx+1)
            plt.imshow(torch.permute(tensors[tensor_idx, slice_idx], (1, 2, 0))) # (C, W, H) -> (W, H, C)
            plt.title(title_list[tensor_idx]) if slice_idx == 0 else None
            plt.xticks([])
            plt.yticks([])
        plt.show()


## Losses
# - weighted mean IoU
# - unweighted mean IoU

class MeanIoU:
    def __init__(self, num_classes, proportions=None, ep=0.0001):
        self.proportions = proportions
        self.ep = ep
        self.num_classes = num_classes

    def __call__(self, preds, label):
            # both are (N, num_classes, W, H) (one hot encoded classes)
        total_iou = 0.0
        preds = torch.permute(preds, (1, 0, 2, 3)) # (C, N, W, H)
        label = torch.permute(label, (1, 0, 2, 3)) # (C, N, W, H)
        for i in range(self.num_classes):
            pred = preds[i].reshape(1, -1) # (1, N*W*H) flatten
            true = label[i].reshape(1, -1) # (1, N*W*H)
            intersection = torch.sum(pred * true, dim=1) # (1)
            union = torch.sum(pred.pow(2), dim=1) + torch.sum(true, dim=1) # (1)
            iou = (intersection + self.ep) / (union + self.ep)

            if self.proportions is not None: iou = iou * (100 / (self.proportions[i] * self.num_classes))

            total_iou += iou
        return total_iou / self.num_classes

class OneHotCrossEntropy:
    def __init__(self, num_classes, device, proportions=None):
        """
        proportions is in the form of python dict.
        """
        self.device = device
        self.num_classes = num_classes
        self.proportions = proportions
        self.weight = self.get_weight() if proportions is not None else None

    def get_weight(self):
        sum = 0
        weight = torch.zeros(self.num_classes, dtype=torch.float32)
        for key, val in self.proportions.items():
            weight[key] = 100 / (self.proportions[key] * self.num_classes)
        return weight

    def __call__(self, preds, label):
        _, label = label.max(dim=1)
        if self.proportions is not None: return nn.CrossEntropyLoss(weight=self.weight.to(device=self.device))(preds, label)
        else: return nn.CrossEntropyLoss()(preds, label)

def train_val_test(model, train_loader, test_loader, val_loader, train_loss_fn, optimizer, epochs,
                   val_loss_fn=None, scheduler=None, patch_size=None, color_map=None,
                   invert_loss=True, device="cpu", verbose=False, model_file_name=None, optimize_memory=False,
                   grad_accum=None, patch_stride=None, num_classes=6):

    if val_loss_fn is None:
        val_loss_fn = train_loss_fn

    train_metrics = {"loss" : [], "score" : []}
    val_metrics = {"loss" : [], "score" : []}

    best_val_loss = 2000000000

    if model_file_name is None:
        print(f"train-val start, model will not be saved (model_file_name is not set)")
    else:
        print(f"train-val start, model will be saved as {model_file_name}")
    for epoch_idx in range(epochs):
        print(f"epoch {epoch_idx} - training")
        train_loss = train_one_epoch(model, train_loader, train_loss_fn,
                                     optimizer, patch_size, color_map, invert_loss,
                                     device, verbose, optimize_memory, grad_accum,
                                     patch_stride, num_classes)

        print(f"epoch {epoch_idx} - testing")
        val_loss = val_one_epoch(model, val_loader, val_loss_fn,
                                 patch_size, color_map, invert_loss,
                                 device, verbose, patch_stride, num_classes)

        if model_file_name is not None:
            if val_loss < best_val_loss: # save better model, lower loss is better
                torch.save(model.state_dict(), model_file_name)
                best_val_loss = val_loss

        train_metrics["loss"].append(train_loss)
        val_metrics["loss"].append(val_loss)

        train_metrics["score"].append(1 - train_loss)
        val_metrics["score"].append(1 - val_loss)
        if scheduler is not None: scheduler.step(val_loss)

    print("training complete. displaying loss graph...")

    plt.figure(figsize=(12, 8))
    plt.plot(train_metrics["loss"], label="Train loss")
    plt.plot(val_metrics["loss"], label="Validation loss")
    plt.title("Loss (IoU)")
    plt.legend()
    plt.xlabel("Epochs")
    plt.ylabel("Intersection over Union (IoU)")
    plt.show()

    print("Testing model...")

    test_metrics, imgs, labels, preds = test_one_epoch(model, test_loader, val_loss_fn,
                                                    patch_size, color_map, invert_loss,
                                                       device, verbose, patch_stride, num_classes, 0)
    print(f"Test loss: {test_metrics['loss']}")
    print(f"Test score: {test_metrics['score']}")
    print(f"Test recall: {test_metrics['recall']}")
    print(f"Test F1-score: {test_metrics['f1']}")

    display_images({
        "Image" : imgs,
        "Label" : labels,
        "Predictions" : preds
    }, image_size=(12, 12))

    return train_metrics, val_metrics, test_metrics

class RandomRotationFromList:
    def __init__(self, degrees):
        self.degrees = degrees

    def __call__(self, img):
        angle = self.degrees[torch.randint(len(self.degrees), size=(1,))]
        return T.functional.rotate(img, angle)

In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(DEVICE)

cuda


### Model define

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision

class DS_Conv2d(nn.Module):
    def __init__(self, in_channels, out_channels, filters_per_layer=1, **kwargs):
        super(DS_Conv2d, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels*filters_per_layer , groups=in_channels,**kwargs)
        self.pointwise = nn.Conv2d(in_channels*filters_per_layer, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, out_channels_per_conv=-1, reduced_channels=None):
        super(ASPP, self).__init__()
        if out_channels_per_conv == -1:
            out_channels_per_conv = out_channels
        self.reduced_channels = reduced_channels
        if self.reduced_channels is not None:
            self.reduce_channels = nn.Conv2d(in_channels, self.reduced_channels, 1, 1, 0)
            in_channels = self.reduced_channels

        self.conv1 = nn.Conv2d(in_channels, out_channels_per_conv, kernel_size=1, stride=1, padding=0, bias=False)
        self.conv2 = DS_Conv2d(in_channels, out_channels_per_conv, kernel_size=3, stride=1, padding=6, dilation=6, bias=False)
        self.conv3 = DS_Conv2d(in_channels, out_channels_per_conv, kernel_size=3, stride=1, padding=12, dilation=12, bias=False)
        self.conv4 = DS_Conv2d(in_channels, out_channels_per_conv, kernel_size=3, stride=1, padding=18, dilation=18, bias=False)
        self.global_avg_pooling = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels_per_conv, kernel_size=1, stride=1, padding=0)
        )
        self.final_conv = DS_Conv2d(out_channels_per_conv*5, out_channels, kernel_size=1, bias=False)

    def forward(self, x):
        if self.reduced_channels is not None:
            x = self.reduce_channels(x)
        x1 = self.conv1(x)
        x2 = self.conv2(x)
        x3 = self.conv3(x)
        x4 = self.conv4(x)
        x5 = self.global_avg_pooling(x)
        x5 = F.interpolate(x5, size=x.shape[-2:], mode="bilinear", align_corners=True)
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1) # concat along channel dim
        return self.final_conv(x_cat)

class DeepLabV3PlusDecoder(nn.Module):
    def __init__(self, low_level_ftr_channels, out_channels, latent_channels, aspp_out_channels):
        super(DeepLabV3PlusDecoder, self).__init__()
        self.conv1x1 = nn.Conv2d(low_level_ftr_channels, latent_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn_1 = nn.BatchNorm2d(latent_channels)
        self.conv3x3_1 = DS_Conv2d(latent_channels+aspp_out_channels, latent_channels+aspp_out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn_2 = nn.BatchNorm2d(latent_channels+aspp_out_channels)
        self.conv3x3_2 = DS_Conv2d(latent_channels+aspp_out_channels, latent_channels+aspp_out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn_3= nn.BatchNorm2d(latent_channels+aspp_out_channels)

        self.classifier = DS_Conv2d(latent_channels+aspp_out_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x, aspp_output):
        x = self.conv1x1(x)
        x - self.bn_1(x)
        x = self.relu(x)

        aspp_output = F.interpolate(aspp_output, scale_factor=4, mode='bilinear', align_corners=True)
        x = torch.cat([x, aspp_output], dim=1)

        x = self.conv3x3_1(x)
        x = self.bn_2(x)
        x = self.relu(x)

        x = self.conv3x3_2(x)
        x = self.bn_3(x)
        x = self.relu(x)
        x = self.classifier(x)

        return x # prediction

class DeepLabV3Plus(nn.Module):
    def __init__(self, in_channels, out_channels, pretrained=True, freeze_backbone=False):
        super(DeepLabV3Plus, self).__init__()
        self.low_level_channels = 256
        self.backbone = torchvision.models.resnet50(pretrained)
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        if in_channels != 3:
            self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        ####### ResNet50 Modifications ###########
        # Modify first conv to be depthwise separable
        # Remove downsampling in layer3 & layer4 (stride=1 instead of 2)

        # self.backbone.layer3[0].conv2.stride = (1, 1)
        # self.backbone.layer3[0].downsample[0].stride = (1, 1)
        # Add dilation to keep the receptive field large
        self.backbone.layer3[0].conv2.dilation = (2, 2)
        self.backbone.layer3[0].conv2.padding = (2, 2)

        self.backbone.layer4[0].conv2.stride = (1, 1)
        self.backbone.layer4[0].downsample[0].stride = (1, 1)
        # Add dilation to keep the receptive field large
        self.backbone.layer4[0].conv2.dilation = (2, 2)
        self.backbone.layer4[0].conv2.padding = (2, 2)

        #############################################
        self.aspp_out_channels = 128
        self.aspp = ASPP(2048, self.aspp_out_channels, reduced_channels=512)
        self.decoder = DeepLabV3PlusDecoder(self.low_level_channels, out_channels, 128, self.aspp_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        size = x.shape[-2:]
        # Encoder
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        low_level_features = self.backbone.layer1(x)

        x = self.backbone.layer2(low_level_features)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)

        x = self.aspp(x)
        x = self.decoder(low_level_features, x)
        x = F.interpolate(x, scale_factor=4, mode="bilinear", align_corners=False)
        return x

### Data parameters

In [ ]:
###### Data parameters ######
# Loader
TRAIN_RATIO = 0.8
VAL_TEST_RATIO = 0.5
BATCH_SIZE = 4
NUM_WORKERS = 4
SHUFFLE = True

# Image Patching and Transforms
PATCH_SIZE = 512
PATCH_STRIDE = 256 #(based on https://www.tandfonline.com/doi/full/10.1080/17538947.2020.1831087#d1e1625)
PAIR_TRANSFORMS = T.Compose([
    T.RandomVerticalFlip(p=0.5),
    T.RandomHorizontalFlip(p=0.5),
    RandomRotationFromList([0, 90, 180, 270])
])
X_TRAIN_TRANSFORMS = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)
])
Y_TRAIN_TRANSFORMS = None

PAIR_EVAL_TRANSFORMS = None
X_EVAL_TRANSFORMS = None
Y_EVAL_TRANSFORMS = None
# pair_transform will be applied first

COLOR_MAP = {
    (255, 255, 255) : 0, # impervious
    (0, 0, 255) : 1, # building
    (0, 255, 255) : 2, # low vegetation
    (0, 255, 0) : 3, # tree
    (255, 255, 0) : 4, # car
    (255, 0, 0) : 5 # clutter/background
}
### Randomization Seed
SEED = 42
torch.manual_seed(SEED)

PROPORTIONS = {
        0: 31.35,
        1: 24.81,
        2: 22.07,
        3: 15.31,
        4: 1.71,
        5: 4.75
}

### Training hyperparameters

In [ ]:
# Hyperparameters
## Model definition
in_channels = 3
out_channels = 6
pretrained = False
model = DeepLabV3Plus(in_channels, out_channels, pretrained=pretrained)
#####

# Training params
LEARNING_RATE = 1e-4
EPOCHS = 10
NUM_BATCH_GRAD_ACCUM = 1
OPTIMIZE_MEMORY = True


TRAINING_LOSS = MeanIoU(6, PROPORTIONS)
VAL_TEST_LOSS = MeanIoU(6, PROPORTIONS)
# if none, will be set to training_loss
INVERT_LOSS = True
# if loss is computed as 1 - loss_fn(), set to True

OPTIMIZER = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
SCHEDULER = torch.optim.lr_scheduler.ReduceLROnPlateau(OPTIMIZER, mode="min", factor=0.1)

MODEL_FILE_NAME = "deeplabv3plus_resnet_random_adam.pth"
VERBOSE = True

TRAIN_METRICS_FILE_NAME = "dlv3plus_resnet_random_adam_train.json"
TEST_METRICS_FILE_NAME = "dlv3plus_resnet_random_adam_test.json"
VAL_METRICS_FILE_NAME = "dlv3plus_resnet_random_adam_val.json"

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


### Load images and labels from folder

In [ ]:
DATASET_FOLDER_PATH = "./drive/MyDrive/Transfer-Learning-LULC/Potsdam.zip"
DEST = "."
if "Potsdam.zip" not in os.listdir(DEST):
    subprocess.run(["cp", DATASET_FOLDER_PATH, DEST])
    subprocess.run(["unzip", "Potsdam.zip"])
    subprocess.run(["mkdir", "-p", "./images"])
    subprocess.run(["unzip", "./Potsdam/2_Ortho_RGB.zip", "-d", "./images"])
    subprocess.run(["mkdir", "-p", "./labels"])
    subprocess.run(["unzip", "./Potsdam/5_Labels_all.zip", "-d", "./labels"])

In [ ]:
import os
import numpy as np

img_path = "./images/2_Ortho_RGB/"
label_path = "./labels/"
X = []
y = []
# Load images
for file_path in os.listdir(img_path):
    if file_path[-3:] == "tif":
        X.append(img_path + file_path)
        y.append(label_path + file_path[:-7] + "label.tif")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=TRAIN_RATIO, random_state=SEED)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, train_size=VAL_TEST_RATIO, random_state=SEED)

train_dataset = LCDataset(X_train, y_train, PAIR_TRANSFORMS, X_TRAIN_TRANSFORMS, Y_TRAIN_TRANSFORMS)
test_dataset = LCDataset(X_test, y_test, PAIR_EVAL_TRANSFORMS, X_EVAL_TRANSFORMS, Y_EVAL_TRANSFORMS)
val_dataset = LCDataset(X_val, y_val, PAIR_EVAL_TRANSFORMS, Y_EVAL_TRANSFORMS, Y_EVAL_TRANSFORMS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE, num_workers=NUM_WORKERS)

### Train-val-test

In [ ]:
train_metrics, val_metrics, test_metrics = train_val_test(model, train_loader, test_loader, val_loader, TRAINING_LOSS, OPTIMIZER,
                                                          epochs=EPOCHS, val_loss_fn=VAL_TEST_LOSS, scheduler=SCHEDULER,
                                                          patch_size=PATCH_SIZE, color_map=COLOR_MAP, invert_loss=INVERT_LOSS,
                                                          device=DEVICE, verbose=VERBOSE, model_file_name=MODEL_FILE_NAME,
                                                          optimize_memory=OPTIMIZE_MEMORY, grad_accum=NUM_BATCH_GRAD_ACCUM,
                                                          patch_stride=PATCH_STRIDE)

train-val start, model will be saved as deeplabv3plus_resnet_random_adam.pth
epoch 0 - training


Training: 100%|██████████| 8/8 [07:34<00:00, 56.75s/it, loss=0.8972]


epoch 0 - testing


Validating: 100%|██████████| 1/1 [00:31<00:00, 31.26s/it, loss=0.7527]


epoch 1 - training


Training:   0%|          | 0/8 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
with open(TRAIN_METRICS_FILE_NAME, "w") as f:
    json.dump(train_metrics, f, indent=4)

with open(TEST_METRICS_FILE_NAME, "w") as f:
    json.dump(test_metrics, f, indent=4)

with open(VAL_METRICS_FILE_NAME, "w") as f:
    json.dump(val_metrics, f, indent=4)

In [ ]:
subprocess.run(["cp", MODEL_FILE_NAME, "./drive/MyDrive/Transfer-Learning-LULC/model_parameters"])
subprocess.run(["cp", TRAIN_METRICS_FILE_NAME, "./drive/MyDrive/Transfer-Learning-LULC/metrics"])
subprocess.run(["cp", TEST_METRICS_FILE_NAME, "./drive/MyDrive/Transfer-Learning-LULC/metrics"])
subprocess.run(["cp", VAL_METRICS_FILE_NAME, "./drive/MyDrive/Transfer-Learning-LULC/metrics"])

In [ ]:
free_gpu()